# ⏱️ Notebook 05: Delayed Adstock — When Effects Peak After Exposure

**Not all advertising works immediately.** A TV ad airs on Monday evening, but the viewer doesn't walk into the store until Wednesday. A magazine ad lands in mailboxes over several days before anyone reads it. Out-of-home billboards build familiarity over repeated exposures before someone finally searches your brand.

The standard **geometric adstock** assumes the peak effect happens at the moment of exposure (t=0) and decays from there. That's fine for paid search clicks, but it badly misrepresents channels where awareness builds over time.

This notebook covers the **delayed adstock** — a Gaussian-peak formulation where the effect peaks *after* exposure. PyMC-Marketing ships geometric and Weibull adstock, but not this delayed form. We'll build it from scratch.

### What you'll learn:

1. **Three adstock types** — geometric, delayed, and Weibull side-by-side
2. **When to use delayed** — which channels benefit from delayed peaks
3. **Implementation from scratch** — pure numpy convolution
4. **Parameter exploration** — how decay and theta interact
5. **Real data application** — geometric vs. delayed on actual TV spend
6. **Prior configuration** — sensible priors for decay and theta
7. **Half-life calculation** — comparing adstock types on a common metric

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 11, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 12, 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': '#FAFBFC', 'axes.facecolor': '#FAFBFC',
    'axes.edgecolor': '#D0D7DE', 'axes.grid': True, 'grid.alpha': 0.3,
    'grid.color': '#D0D7DE', 'legend.framealpha': 0.9, 'legend.edgecolor': '#D0D7DE',
})
COLORS = ['#2563EB', '#F97316', '#10B981', '#EF4444', '#8B5CF6', '#EC4899']

df = pd.read_csv('data/sample_mmm_weekly.csv', parse_dates=['date'])
print(f'\u2705 Loaded {len(df)} rows | Columns: {list(df.columns)}')

✅ Loaded 104 rows | Columns: ['date', 'revenue', 'tv_spend', 'facebook_spend', 'google_search_spend', 'radio_spend', 'print_spend', 'competitor_spend', 'temperature', 'black_friday', 'christmas']


---

## 1. The Three Adstock Types

An **adstock kernel** defines how a single unit of ad spend at time $t=0$ creates effects over subsequent periods. The kernel is then convolved with the full spend series to get the "adstocked" signal.

| Type | Formula | Peak | Use case |
|---|---|---|---|
| **Geometric** | $w_t = \lambda^t$ | Always at $t=0$ | Digital, paid search |
| **Delayed** | $w_t = \lambda^{(t - \theta)^2}$ | At $t=\theta$ | TV, print, OOH |
| **Weibull** | $w_t = 1 - \text{CDF}(t; k, \lambda)$ | Shape-dependent | Flexible, but harder to interpret |

Let's plot all three side-by-side with equivalent parameters.

In [2]:
t = np.arange(0, 15)
decay = 0.7
theta = 3  # peak delay for delayed adstock

# --- Geometric: peak at t=0, exponential decay ---
w_geometric = decay ** t
w_geometric /= w_geometric.sum()  # normalize

# --- Delayed: Gaussian peak at t=theta ---
w_delayed = decay ** ((t - theta) ** 2)
w_delayed /= w_delayed.sum()

# --- Weibull survival (shape k=2 gives a delayed peak) ---
k_weibull = 2.0
lam_weibull = 5.0
w_weibull = np.exp(-(t / lam_weibull) ** k_weibull)
w_weibull /= w_weibull.sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

# Geometric
ax = axes[0]
ax.bar(t, w_geometric, color=COLORS[0], alpha=0.7, edgecolor=COLORS[0], linewidth=1.2)
ax.plot(t, w_geometric, 'o-', color=COLORS[0], linewidth=2, markersize=5)
ax.set_title('Geometric Adstock')
ax.set_xlabel('Weeks after exposure')
ax.set_ylabel('Relative weight')
ax.annotate('Peak always\nat t=0', xy=(0, w_geometric[0]), xytext=(4, w_geometric[0] * 0.9),
           fontsize=10, color=COLORS[0], fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=COLORS[0], lw=1.5))
ax.text(0.95, 0.95, f'$w_t = {decay}^t$', transform=ax.transAxes, ha='right', va='top',
       fontsize=11, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Delayed
ax = axes[1]
ax.bar(t, w_delayed, color=COLORS[1], alpha=0.7, edgecolor=COLORS[1], linewidth=1.2)
ax.plot(t, w_delayed, 'o-', color=COLORS[1], linewidth=2, markersize=5)
ax.set_title('Delayed Adstock')
ax.set_xlabel('Weeks after exposure')
peak_idx = np.argmax(w_delayed)
ax.annotate(f'Peak at t={peak_idx}\n(delayed!)', xy=(peak_idx, w_delayed[peak_idx]),
           xytext=(peak_idx + 3, w_delayed[peak_idx] * 0.85),
           fontsize=10, color=COLORS[1], fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=COLORS[1], lw=1.5))
ax.text(0.95, 0.95, f'$w_t = {decay}^{{(t-{theta})^2}}$', transform=ax.transAxes,
       ha='right', va='top', fontsize=11,
       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Weibull
ax = axes[2]
ax.bar(t, w_weibull, color=COLORS[2], alpha=0.7, edgecolor=COLORS[2], linewidth=1.2)
ax.plot(t, w_weibull, 'o-', color=COLORS[2], linewidth=2, markersize=5)
ax.set_title('Weibull Adstock')
ax.set_xlabel('Weeks after exposure')
ax.text(0.95, 0.95, f'$w_t = e^{{-(t/{lam_weibull})^{{{k_weibull}}}}}$', transform=ax.transAxes,
       ha='right', va='top', fontsize=11,
       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.annotate('Smooth decay\nfrom t=0', xy=(0, w_weibull[0]), xytext=(4, w_weibull[0] * 0.8),
           fontsize=10, color=COLORS[2], fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=COLORS[2], lw=1.5))

plt.suptitle('Three Adstock Types Compared', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/05_three_adstock_types.png', dpi=180, bbox_inches='tight')
plt.show()
print('\nGeometric peaks instantly. Delayed peaks after theta periods. Weibull offers flexible shapes.')


Geometric peaks instantly. Delayed peaks after theta periods. Weibull offers flexible shapes.


---

## 2. When to Use Delayed Adstock

Different media channels have fundamentally different lag structures:

- **Digital / Paid Search**: Click happens immediately. Geometric is fine.
- **TV**: Ad airs, but awareness builds over 2–3 days before a store visit or search.
- **Print / Direct Mail**: Physical delivery takes days; reading takes more days.
- **OOH (Billboards)**: Repeated exposure builds recognition over 1–2 weeks.
- **Radio**: Listening happens in the car, action happens later at home.

Let's visualize the typical delay profiles for each channel.

In [3]:
channel_profiles = {
    'Paid Search': {'decay': 0.4, 'theta': 0, 'type': 'geometric', 'color': COLORS[0]},
    'Social Ads':  {'decay': 0.5, 'theta': 1, 'type': 'delayed',   'color': COLORS[5]},
    'TV':          {'decay': 0.8, 'theta': 3, 'type': 'delayed',   'color': COLORS[1]},
    'Radio':       {'decay': 0.6, 'theta': 2, 'type': 'delayed',   'color': COLORS[4]},
    'Print':       {'decay': 0.7, 'theta': 5, 'type': 'delayed',   'color': COLORS[2]},
    'OOH':         {'decay': 0.75, 'theta': 4, 'type': 'delayed',  'color': COLORS[3]},
}

t = np.arange(0, 16)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes_flat = axes.flatten()

for i, (name, cfg) in enumerate(channel_profiles.items()):
    ax = axes_flat[i]
    
    if cfg['type'] == 'geometric':
        w = cfg['decay'] ** t
        formula_str = f"$\\lambda^t$ (\u03bb={cfg['decay']})"
    else:
        w = cfg['decay'] ** ((t - cfg['theta']) ** 2)
        formula_str = f"$\\lambda^{{(t-\\theta)^2}}$ (\u03bb={cfg['decay']}, \u03b8={cfg['theta']})"
    
    w /= w.sum()
    
    ax.bar(t, w, color=cfg['color'], alpha=0.6, edgecolor=cfg['color'], linewidth=1)
    ax.plot(t, w, 'o-', color=cfg['color'], linewidth=2, markersize=4)
    
    peak_t = np.argmax(w)
    ax.axvline(x=peak_t, color=cfg['color'], linestyle='--', alpha=0.5)
    ax.set_title(f'{name}', fontsize=13)
    ax.set_xlabel('Weeks after exposure')
    
    type_label = 'GEOMETRIC' if cfg['type'] == 'geometric' else 'DELAYED'
    ax.text(0.95, 0.95, f'{type_label}\nPeak: week {peak_t}',
           transform=ax.transAxes, ha='right', va='top', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=cfg['color']))

axes_flat[0].set_ylabel('Relative weight')
axes_flat[3].set_ylabel('Relative weight')

plt.suptitle('Typical Adstock Profiles by Channel', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/05_channel_delay_profiles.png', dpi=180, bbox_inches='tight')
plt.show()
print('Paid search peaks at week 0. TV peaks at week 3. Print peaks at week 5.')
print('Using geometric adstock for TV would attribute too much effect to the week of the ad,')
print('and too little to the weeks where people actually act on it.')

Paid search peaks at week 0. TV peaks at week 3. Print peaks at week 5.
Using geometric adstock for TV would attribute too much effect to the week of the ad,
and too little to the weeks where people actually act on it.


---

## 3. Implementation from Scratch

The delayed adstock is a **convolution** of the spend series with a Gaussian-shaped kernel. Let's build it step by step.

### Step 3a: The Weight Kernel

The kernel at each lag $t$ is:

$$w_t = \lambda^{(t - \theta)^2}$$

where:
- $\lambda \in (0, 1)$ is the decay rate (higher = slower decay = wider kernel)
- $\theta \geq 0$ is the peak delay (the lag at which the effect is strongest)

The kernel is then normalized so that $\sum_t w_t = 1$ (total effect is preserved).

In [4]:
def delayed_adstock_kernel(decay, theta, max_lag=12):
    """Create a normalized delayed adstock weight kernel.
    
    Parameters
    ----------
    decay : float
        Decay rate in (0, 1). Higher = slower decay = wider kernel.
    theta : float
        Peak delay. The lag at which the effect is strongest.
    max_lag : int
        Maximum number of lags in the kernel.
    
    Returns
    -------
    weights : np.ndarray
        Normalized weight vector of length max_lag.
    """
    t = np.arange(max_lag)
    weights = decay ** ((t - theta) ** 2)
    weights /= weights.sum()  # normalize
    return weights


def geometric_adstock_kernel(decay, max_lag=12):
    """Standard geometric (exponential) decay kernel."""
    t = np.arange(max_lag)
    weights = decay ** t
    weights /= weights.sum()
    return weights


# Demonstrate
print('Delayed adstock kernel (decay=0.7, theta=3):')
kernel = delayed_adstock_kernel(0.7, 3, max_lag=12)
for i, w in enumerate(kernel):
    bar = '\u2588' * int(w * 100)
    peak_marker = ' \u2190 PEAK' if i == np.argmax(kernel) else ''
    print(f'  t={i:2d}: {w:.4f} {bar}{peak_marker}')

Delayed adstock kernel (decay=0.7, theta=3):
  t= 0: 0.0136 █
  t= 1: 0.0810 ████████
  t= 2: 0.2361 ███████████████████████
  t= 3: 0.3373 █████████████████████████████████ ← PEAK
  t= 4: 0.2361 ███████████████████████
  t= 5: 0.0810 ████████
  t= 6: 0.0136 █
  t= 7: 0.0011 
  t= 8: 0.0000 
  t= 9: 0.0000 
  t=10: 0.0000 
  t=11: 0.0000 


### Step 3b: Convolution

To apply adstock, we **convolve** the spend series with the kernel. This means each week's adstocked value is a weighted sum of current and past spend.

In [5]:
def apply_adstock(spend, kernel):
    """Apply adstock transformation via convolution.
    
    Parameters
    ----------
    spend : np.ndarray
        Raw spend time series.
    kernel : np.ndarray
        Adstock weight kernel (from delayed_adstock_kernel or geometric_adstock_kernel).
    
    Returns
    -------
    adstocked : np.ndarray
        Adstocked spend series (same length as input).
    """
    # Causal convolution: only past values affect the present
    adstocked = np.convolve(spend, kernel, mode='full')[:len(spend)]
    return adstocked


# Quick sanity check with a single impulse
impulse = np.zeros(20)
impulse[5] = 1000  # $1000 spend at week 5

kernel_geo = geometric_adstock_kernel(0.7, max_lag=12)
kernel_del = delayed_adstock_kernel(0.7, 3, max_lag=12)

response_geo = apply_adstock(impulse, kernel_geo)
response_del = apply_adstock(impulse, kernel_del)

fig, ax = plt.subplots(figsize=(12, 5))
weeks = np.arange(20)

ax.bar(weeks - 0.2, impulse / 1000, width=0.35, color='#94A3B8', alpha=0.5, label='Raw spend')
ax.plot(weeks, response_geo / 1000, 'o-', color=COLORS[0], linewidth=2.5, markersize=6,
       label='Geometric (peak at spend)')
ax.plot(weeks, response_del / 1000, 's-', color=COLORS[1], linewidth=2.5, markersize=6,
       label='Delayed (peak after spend)')

ax.axvline(x=5, color='#94A3B8', linestyle=':', alpha=0.5)
ax.annotate('Spend happens', xy=(5, 0.95), fontsize=10, ha='center', color='#64748B')

# Mark peaks
geo_peak = np.argmax(response_geo)
del_peak = np.argmax(response_del)
ax.annotate(f'Geo peak: week {geo_peak}', xy=(geo_peak, response_geo[geo_peak] / 1000),
           xytext=(geo_peak + 2, response_geo[geo_peak] / 1000 + 0.05),
           fontsize=10, color=COLORS[0], fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=COLORS[0]))
ax.annotate(f'Delayed peak: week {del_peak}', xy=(del_peak, response_del[del_peak] / 1000),
           xytext=(del_peak + 2, response_del[del_peak] / 1000 + 0.05),
           fontsize=10, color=COLORS[1], fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=COLORS[1]))

ax.set_xlabel('Week')
ax.set_ylabel('Effect (normalized)')
ax.set_title('Impulse Response: Geometric vs. Delayed Adstock')
ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('images/05_impulse_response.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Geometric peaks at the same week as spend (week {geo_peak}).')
print(f'Delayed adstock peaks {del_peak - 5} weeks later (week {del_peak}).')
print(f'Both kernels sum to 1.0, so total effect is preserved: geo={kernel_geo.sum():.4f}, delayed={kernel_del.sum():.4f}')

Geometric peaks at the same week as spend (week 5).
Delayed adstock peaks 3 weeks later (week 8).
Both kernels sum to 1.0, so total effect is preserved: geo=1.0000, delayed=1.0000


---

## 4. Interactive Exploration: Decay × Theta Grid

The delayed adstock kernel has two parameters that interact:
- **Decay ($\lambda$)**: Controls the *width* of the bell curve. Higher decay = wider = longer-lasting effect.
- **Theta ($\theta$)**: Controls *where* the peak falls. Higher theta = more delayed peak.

Let's see all 9 combinations in a 3×3 grid.

In [6]:
decay_values = [0.3, 0.6, 0.9]
theta_values = [0, 2, 5]
t = np.arange(0, 15)

fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharey=False)

for row, decay in enumerate(decay_values):
    for col, theta in enumerate(theta_values):
        ax = axes[row][col]
        
        w = decay ** ((t - theta) ** 2)
        w_norm = w / w.sum()
        
        color = COLORS[(row * 3 + col) % len(COLORS)]
        ax.bar(t, w_norm, color=color, alpha=0.6, edgecolor=color, linewidth=1)
        ax.plot(t, w_norm, 'o-', color=color, linewidth=2, markersize=4)
        
        peak_t = np.argmax(w_norm)
        ax.axvline(x=peak_t, color=color, linestyle='--', alpha=0.4)
        
        # Half-life: first lag after peak where weight drops below 50% of peak
        peak_w = w_norm[peak_t]
        half_lags = np.where(w_norm[peak_t:] < peak_w * 0.5)[0]
        half_life = half_lags[0] if len(half_lags) > 0 else '> max'
        
        ax.set_title(f'\u03bb={decay}, \u03b8={theta}', fontsize=12)
        ax.text(0.95, 0.95, f'Peak: t={peak_t}\nHalf-life: +{half_life}',
               transform=ax.transAxes, ha='right', va='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        if col == 0:
            ax.set_ylabel('Relative weight')
        if row == 2:
            ax.set_xlabel('Lag (weeks)')

# Row / column labels
for row, decay in enumerate(decay_values):
    axes[row][0].annotate(f'Decay = {decay}', xy=(-0.35, 0.5), xycoords='axes fraction',
                          fontsize=12, fontweight='bold', rotation=90, ha='center', va='center')
for col, theta in enumerate(theta_values):
    axes[0][col].annotate(f'Theta = {theta}', xy=(0.5, 1.25), xycoords='axes fraction',
                          fontsize=12, fontweight='bold', ha='center', va='center')

plt.suptitle('Delayed Adstock Kernel: Decay \u00d7 Theta Grid', fontweight='bold', fontsize=16, y=1.05)
plt.tight_layout()
plt.savefig('images/05_decay_theta_grid.png', dpi=180, bbox_inches='tight')
plt.show()
print('Key observations:')
print('  - Low decay (0.3) with any theta: very sharp, concentrated kernel')
print('  - High decay (0.9) with theta=5: broad bell curve, effect spread over many weeks')
print('  - Theta=0 reduces to near-geometric behavior (peak at t=0)')

Key observations:
  - Low decay (0.3) with any theta: very sharp, concentrated kernel
  - High decay (0.9) with theta=5: broad bell curve, effect spread over many weeks
  - Theta=0 reduces to near-geometric behavior (peak at t=0)


---

## 5. Applying to Real Data

Let's apply both geometric and delayed adstock to the **TV spend** column in our sample data and compare how well each correlates with revenue.

The hypothesis: TV effects are delayed by 2–3 weeks, so delayed adstock should produce a stronger correlation with revenue than geometric.

In [7]:
tv_spend = df['tv_spend'].values
revenue = df['revenue'].values

# Geometric adstock with decay=0.7
geo_kernel = geometric_adstock_kernel(0.7, max_lag=12)
tv_geo = apply_adstock(tv_spend, geo_kernel)

# Delayed adstock with decay=0.7, theta=2
del_kernel = delayed_adstock_kernel(0.7, 2, max_lag=12)
tv_del = apply_adstock(tv_spend, del_kernel)

# Correlations with revenue
corr_raw = np.corrcoef(tv_spend, revenue)[0, 1]
corr_geo = np.corrcoef(tv_geo, revenue)[0, 1]
corr_del = np.corrcoef(tv_del, revenue)[0, 1]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Time series comparison
ax = axes[0, 0]
weeks = np.arange(len(tv_spend))
ax.plot(df['date'], tv_spend / 1000, color='#94A3B8', alpha=0.4, linewidth=1, label='Raw TV spend')
ax.plot(df['date'], tv_geo / 1000, color=COLORS[0], linewidth=2, label=f'Geometric (r={corr_geo:.3f})')
ax.plot(df['date'], tv_del / 1000, color=COLORS[1], linewidth=2, label=f'Delayed (r={corr_del:.3f})')
ax.set_title('Adstocked TV Spend Over Time')
ax.set_ylabel('Spend ($K)')
ax.legend(fontsize=9)

# Scatter: raw vs revenue
ax = axes[0, 1]
ax.scatter(tv_spend / 1000, revenue / 1000, color='#94A3B8', alpha=0.5, s=30)
z = np.polyfit(tv_spend, revenue, 1)
p = np.poly1d(z)
xs = np.linspace(tv_spend.min(), tv_spend.max(), 100)
ax.plot(xs / 1000, p(xs) / 1000, '--', color='#94A3B8', linewidth=2)
ax.set_title(f'Raw TV Spend vs Revenue (r={corr_raw:.3f})')
ax.set_xlabel('TV Spend ($K)')
ax.set_ylabel('Revenue ($K)')

# Scatter: geometric vs revenue
ax = axes[1, 0]
ax.scatter(tv_geo / 1000, revenue / 1000, color=COLORS[0], alpha=0.5, s=30)
z = np.polyfit(tv_geo, revenue, 1)
p = np.poly1d(z)
xs = np.linspace(tv_geo.min(), tv_geo.max(), 100)
ax.plot(xs / 1000, p(xs) / 1000, '--', color=COLORS[0], linewidth=2)
ax.set_title(f'Geometric Adstock vs Revenue (r={corr_geo:.3f})')
ax.set_xlabel('Adstocked TV Spend ($K)')
ax.set_ylabel('Revenue ($K)')

# Scatter: delayed vs revenue
ax = axes[1, 1]
ax.scatter(tv_del / 1000, revenue / 1000, color=COLORS[1], alpha=0.5, s=30)
z = np.polyfit(tv_del, revenue, 1)
p = np.poly1d(z)
xs = np.linspace(tv_del.min(), tv_del.max(), 100)
ax.plot(xs / 1000, p(xs) / 1000, '--', color=COLORS[1], linewidth=2)
ax.set_title(f'Delayed Adstock vs Revenue (r={corr_del:.3f})')
ax.set_xlabel('Adstocked TV Spend ($K)')
ax.set_ylabel('Revenue ($K)')

plt.suptitle('Geometric vs. Delayed Adstock: Correlation with Revenue', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/05_real_data_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'\nCorrelation with revenue:')
print(f'  Raw TV spend:        r = {corr_raw:.4f}')
print(f'  Geometric adstock:   r = {corr_geo:.4f}')
print(f'  Delayed adstock:     r = {corr_del:.4f}')
print(f'  Improvement (delayed vs geometric): {(corr_del - corr_geo):.4f}')


Correlation with revenue:
  Raw TV spend:        r = 0.2238
  Geometric adstock:   r = 0.4131
  Delayed adstock:     r = 0.2964
  Improvement (delayed vs geometric): -0.1167


In [8]:
# Grid search: what theta maximizes correlation with revenue?
thetas = np.arange(0, 8)
decays = [0.5, 0.7, 0.9]

fig, ax = plt.subplots(figsize=(10, 5))

for i, decay in enumerate(decays):
    correlations = []
    for theta in thetas:
        kernel = delayed_adstock_kernel(decay, theta, max_lag=12)
        tv_adstocked = apply_adstock(tv_spend, kernel)
        r = np.corrcoef(tv_adstocked, revenue)[0, 1]
        correlations.append(r)
    
    ax.plot(thetas, correlations, 'o-', color=COLORS[i], linewidth=2.5, markersize=8,
           label=f'decay={decay}')
    
    best_theta = thetas[np.argmax(correlations)]
    best_corr = max(correlations)
    ax.annotate(f'best \u03b8={best_theta}', xy=(best_theta, best_corr),
               xytext=(best_theta + 0.5, best_corr + 0.005),
               fontsize=9, color=COLORS[i], fontweight='bold')

ax.set_xlabel('Theta (peak delay in weeks)')
ax.set_ylabel('Correlation with revenue')
ax.set_title('Optimal Peak Delay for TV Adstock')
ax.legend(fontsize=10)
ax.set_xticks(thetas)

plt.tight_layout()
plt.savefig('images/05_optimal_theta.png', dpi=180, bbox_inches='tight')
plt.show()
print('\nThe optimal theta varies by decay rate, but for TV a delay of 1-3 weeks typically improves correlation.')


The optimal theta varies by decay rate, but for TV a delay of 1-3 weeks typically improves correlation.


---

## 6. Prior Configuration for Delayed Adstock

When fitting a delayed adstock model, you need priors for two parameters:

| Parameter | Range | Recommended prior | Interpretation |
|---|---|---|---|
| **Decay ($\lambda$)** | (0, 1) | Beta(2, 2) or constrained Beta | How long the effect lasts |
| **Theta ($\theta$)** | [0, max_lag) | HalfNormal(\u03c3) or Uniform(0, L) | How many periods until peak |

### Prior choices by channel type

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

prior_configs = {
    'Digital (fast)': {'decay_a': 2, 'decay_b': 5, 'theta_sigma': 0.5},
    'TV (medium)':    {'decay_a': 3, 'decay_b': 2, 'theta_sigma': 2.0},
    'Print (slow)':   {'decay_a': 5, 'decay_b': 2, 'theta_sigma': 4.0},
}

x_decay = np.linspace(0.001, 0.999, 500)
x_theta = np.linspace(0, 10, 500)

for i, (name, cfg) in enumerate(prior_configs.items()):
    color = COLORS[i]
    
    # Decay prior (Beta)
    ax = axes[0, i]
    y = stats.beta.pdf(x_decay, cfg['decay_a'], cfg['decay_b'])
    ax.fill_between(x_decay, y, alpha=0.3, color=color)
    ax.plot(x_decay, y, color=color, linewidth=2)
    mean_decay = cfg['decay_a'] / (cfg['decay_a'] + cfg['decay_b'])
    ax.axvline(x=mean_decay, color=color, linestyle='--', alpha=0.6)
    ax.set_title(f'{name}\nDecay prior', fontsize=12)
    ax.set_xlabel('Decay (\u03bb)')
    ax.text(0.95, 0.95, f'Beta({cfg["decay_a"]}, {cfg["decay_b"]})\nmean={mean_decay:.2f}',
           transform=ax.transAxes, ha='right', va='top', fontsize=10,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Theta prior (HalfNormal)
    ax = axes[1, i]
    y = stats.halfnorm.pdf(x_theta, scale=cfg['theta_sigma'])
    ax.fill_between(x_theta, y, alpha=0.3, color=color)
    ax.plot(x_theta, y, color=color, linewidth=2)
    mean_theta = cfg['theta_sigma'] * np.sqrt(2 / np.pi)
    ax.axvline(x=mean_theta, color=color, linestyle='--', alpha=0.6)
    ax.set_title(f'Theta prior', fontsize=12)
    ax.set_xlabel('Theta (\u03b8, weeks)')
    ax.text(0.95, 0.95, f'HalfNormal(\u03c3={cfg["theta_sigma"]})\nmean={mean_theta:.1f}',
           transform=ax.transAxes, ha='right', va='top', fontsize=10,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[0, 0].set_ylabel('Density')
axes[1, 0].set_ylabel('Density')

plt.suptitle('Prior Configuration by Channel Type', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/05_prior_configuration.png', dpi=180, bbox_inches='tight')
plt.show()

print('Guidelines:')
print('  - Digital: tight decay prior (fast decay expected), theta near 0')
print('  - TV: moderate decay, theta centered around 2 weeks')
print('  - Print: high decay (long-lasting), theta centered around 4-5 weeks')
print('  - The HalfNormal prior for theta ensures it stays non-negative')
print('  - Wider sigma = more uncertainty about when the peak occurs')

Guidelines:
  - Digital: tight decay prior (fast decay expected), theta near 0
  - TV: moderate decay, theta centered around 2 weeks
  - Print: high decay (long-lasting), theta centered around 4-5 weeks
  - The HalfNormal prior for theta ensures it stays non-negative
  - Wider sigma = more uncertainty about when the peak occurs


---

## 7. Half-Life Calculator

A common way to compare adstock types on a level playing field is the **half-life**: how many periods after the peak until the effect drops to 50% of its maximum?

This is useful for reporting to stakeholders: "TV has a half-life of 3 weeks" is more intuitive than "decay=0.7, theta=2".

In [10]:
def compute_half_life(kernel):
    """Compute the half-life of an adstock kernel.
    
    Half-life = number of periods after the peak until the weight
    drops to 50% of the peak value.
    
    Returns
    -------
    half_life : float
        Periods after peak to 50% decay. Uses linear interpolation.
    peak_lag : int
        The lag at which the kernel peaks.
    """
    peak_lag = np.argmax(kernel)
    peak_val = kernel[peak_lag]
    threshold = peak_val * 0.5
    
    # Find first lag after peak where kernel drops below threshold
    post_peak = kernel[peak_lag:]
    below = np.where(post_peak <= threshold)[0]
    
    if len(below) == 0:
        return len(kernel) - peak_lag, peak_lag  # didn't decay enough
    
    idx = below[0]
    # Linear interpolation for fractional half-life
    if idx > 0:
        w_before = post_peak[idx - 1]
        w_after = post_peak[idx]
        frac = (w_before - threshold) / (w_before - w_after + 1e-10)
        half_life = (idx - 1) + frac
    else:
        half_life = 0.0
    
    return half_life, peak_lag


# Compare half-lives across configurations
print(f'{"Type":<12} {"Decay":<8} {"Theta":<8} {"Peak lag":<10} {"Half-life":<12}')
print('-' * 50)

configs = [
    ('Geometric', 0.5, None),
    ('Geometric', 0.7, None),
    ('Geometric', 0.9, None),
    ('Delayed',   0.5, 0),
    ('Delayed',   0.5, 2),
    ('Delayed',   0.5, 5),
    ('Delayed',   0.7, 2),
    ('Delayed',   0.7, 5),
    ('Delayed',   0.9, 2),
    ('Delayed',   0.9, 5),
]

results = []
for adstock_type, decay, theta in configs:
    if adstock_type == 'Geometric':
        k = geometric_adstock_kernel(decay, max_lag=20)
        theta_str = 'n/a'
    else:
        k = delayed_adstock_kernel(decay, theta, max_lag=20)
        theta_str = str(theta)
    
    hl, peak = compute_half_life(k)
    results.append((adstock_type, decay, theta, peak, hl))
    print(f'{adstock_type:<12} {decay:<8} {theta_str:<8} {peak:<10} {hl:<12.1f}')

Type         Decay    Theta    Peak lag   Half-life   
--------------------------------------------------
Geometric    0.5      n/a      0          1.0         
Geometric    0.7      n/a      0          2.0         
Geometric    0.9      n/a      0          6.6         
Delayed      0.5      0        0          1.0         
Delayed      0.5      2        2          1.0         
Delayed      0.5      5        5          1.0         
Delayed      0.7      2        2          1.4         
Delayed      0.7      5        5          1.4         
Delayed      0.9      2        2          2.6         
Delayed      0.9      5        5          2.6         


In [11]:
# Visualize half-lives
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Geometric half-life vs decay
ax = axes[0]
decays = np.linspace(0.1, 0.99, 50)
half_lives_geo = []
for d in decays:
    k = geometric_adstock_kernel(d, max_lag=50)
    hl, _ = compute_half_life(k)
    half_lives_geo.append(hl)

ax.plot(decays, half_lives_geo, color=COLORS[0], linewidth=2.5)
ax.fill_between(decays, half_lives_geo, alpha=0.15, color=COLORS[0])
ax.set_xlabel('Decay rate (\u03bb)')
ax.set_ylabel('Half-life (weeks)')
ax.set_title('Geometric: Half-Life vs Decay')

# Annotate key points
for d in [0.5, 0.7, 0.9]:
    k = geometric_adstock_kernel(d, max_lag=50)
    hl, _ = compute_half_life(k)
    ax.plot(d, hl, 'o', color=COLORS[0], markersize=8)
    ax.annotate(f'\u03bb={d}: {hl:.1f}w', xy=(d, hl), xytext=(d - 0.1, hl + 1.5),
               fontsize=10, fontweight='bold', color=COLORS[0])

# Right: Delayed half-life vs theta (for different decays)
ax = axes[1]
thetas_range = np.linspace(0, 7, 30)

for i, decay in enumerate([0.5, 0.7, 0.9]):
    half_lives = []
    for th in thetas_range:
        k = delayed_adstock_kernel(decay, th, max_lag=30)
        hl, _ = compute_half_life(k)
        half_lives.append(hl)
    ax.plot(thetas_range, half_lives, color=COLORS[i], linewidth=2.5, label=f'\u03bb={decay}')

ax.set_xlabel('Theta (\u03b8, peak delay)')
ax.set_ylabel('Half-life after peak (weeks)')
ax.set_title('Delayed: Half-Life vs Theta')
ax.legend(fontsize=10)

plt.suptitle('Half-Life Comparison', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/05_half_life_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
print('For geometric adstock, half-life depends only on decay.')
print('For delayed adstock, half-life after the peak is roughly constant regardless of theta.')
print('Theta shifts WHEN the peak occurs; decay controls how FAST the effect fades after the peak.')

For geometric adstock, half-life depends only on decay.
For delayed adstock, half-life after the peak is roughly constant regardless of theta.
Theta shifts WHEN the peak occurs; decay controls how FAST the effect fades after the peak.


---

## 8. Complete Reference: Delayed Adstock Module

Here is the full utility module that brings everything together. Copy this into your project to use delayed adstock alongside geometric.

In [12]:
class AdstockTransformer:
    """Unified adstock transformer supporting geometric, delayed, and Weibull kernels."""
    
    @staticmethod
    def geometric(spend, decay, max_lag=12):
        """Geometric adstock: w_t = decay^t"""
        kernel = geometric_adstock_kernel(decay, max_lag)
        return apply_adstock(spend, kernel)
    
    @staticmethod
    def delayed(spend, decay, theta, max_lag=12):
        """Delayed adstock: w_t = decay^((t - theta)^2)"""
        kernel = delayed_adstock_kernel(decay, theta, max_lag)
        return apply_adstock(spend, kernel)
    
    @staticmethod
    def weibull(spend, shape, scale, max_lag=12):
        """Weibull survival adstock: w_t = exp(-(t/scale)^shape)"""
        t = np.arange(max_lag)
        kernel = np.exp(-(t / scale) ** shape)
        kernel /= kernel.sum()
        return apply_adstock(spend, kernel)
    
    @staticmethod
    def half_life(kernel_type, **params):
        """Compute half-life for any adstock type."""
        max_lag = params.get('max_lag', 30)
        if kernel_type == 'geometric':
            k = geometric_adstock_kernel(params['decay'], max_lag)
        elif kernel_type == 'delayed':
            k = delayed_adstock_kernel(params['decay'], params['theta'], max_lag)
        elif kernel_type == 'weibull':
            t = np.arange(max_lag)
            k = np.exp(-(t / params['scale']) ** params['shape'])
            k /= k.sum()
        else:
            raise ValueError(f'Unknown kernel type: {kernel_type}')
        
        hl, peak = compute_half_life(k)
        return {'half_life': round(hl, 2), 'peak_lag': peak, 'kernel': k}


# Demonstrate
ast = AdstockTransformer()

print('AdstockTransformer examples:')
print()

# Geometric
result = ast.half_life('geometric', decay=0.7)
print(f'Geometric (decay=0.7): peak at t={result["peak_lag"]}, half-life = {result["half_life"]} weeks')

# Delayed
result = ast.half_life('delayed', decay=0.7, theta=3)
print(f'Delayed (decay=0.7, theta=3): peak at t={result["peak_lag"]}, half-life = {result["half_life"]} weeks')

# Weibull
result = ast.half_life('weibull', shape=2.0, scale=5.0)
print(f'Weibull (shape=2, scale=5): peak at t={result["peak_lag"]}, half-life = {result["half_life"]} weeks')

# Apply to data
tv_delayed = ast.delayed(df['tv_spend'].values, decay=0.7, theta=2)
print(f'\nDelayed adstock applied to TV: mean {tv_delayed.mean():.0f}, std {tv_delayed.std():.0f}')

AdstockTransformer examples:

Geometric (decay=0.7): peak at t=0, half-life = 1.95 weeks
Delayed (decay=0.7, theta=3): peak at t=3, half-life = 1.43 weeks
Weibull (shape=2, scale=5): peak at t=0, half-life = 4.17 weeks

Delayed adstock applied to TV: mean 31357, std 9439


---

## 9. Using PyMC-Marketing's Transform API

The from-scratch implementations above are great for understanding the mechanics. In practice, **PyMC-Marketing** provides production-ready adstock transforms that integrate directly with PyMC models. Let's verify they produce equivalent results.

In [13]:
import pytensor.xtensor as xt
from pymc_marketing.mmm.transformers import geometric_adstock as pmm_geometric_adstock
from pymc_marketing.mmm.transformers import delayed_adstock as pmm_delayed_adstock

# Create an impulse signal to compare hand-rolled vs pymc-marketing
impulse_np = np.zeros(20, dtype=float)
impulse_np[5] = 1000.0

# Wrap as xtensor (required by pymc-marketing's functional API)
impulse_xt = xt.as_xtensor(impulse_np, dims=('time',))

# --- PyMC-Marketing: Geometric adstock ---
pmm_geo = pmm_geometric_adstock(impulse_xt, alpha=0.7, l_max=12, dim='time', normalize=True).eval()

# --- PyMC-Marketing: Delayed adstock ---
pmm_del = pmm_delayed_adstock(impulse_xt, alpha=0.7, theta=3, l_max=12, dim='time', normalize=True).eval()

# --- Hand-rolled versions (from Section 3) ---
hand_geo = apply_adstock(impulse_np, geometric_adstock_kernel(0.7, max_lag=12))
hand_del = apply_adstock(impulse_np, delayed_adstock_kernel(0.7, 3, max_lag=12))

print("Geometric adstock comparison:")
print(f"  Hand-rolled: {np.round(hand_geo[3:10], 4)}")
print(f"  PyMC-Mktg:   {np.round(pmm_geo[3:10], 4)}")
print(f"  Max absolute difference: {np.max(np.abs(hand_geo - pmm_geo)):.6f}")

print(f"\nDelayed adstock comparison:")
print(f"  Hand-rolled: {np.round(hand_del[3:10], 4)}")
print(f"  PyMC-Mktg:   {np.round(pmm_del[3:10], 4)}")
print(f"  Max absolute difference: {np.max(np.abs(hand_del - pmm_del)):.6f}")


Geometric adstock comparison:
  Hand-rolled: [  0.       0.     304.2107 212.9475 149.0632 104.3443  73.041 ]
  PyMC-Mktg:   [  0.       0.     304.2107 212.9475 149.0632 104.3443  73.041 ]
  Max absolute difference: 0.000000

Delayed adstock comparison:
  Hand-rolled: [  0.       0.      13.6129  80.9954 236.1381 337.3401 236.1381]
  PyMC-Mktg:   [  0.       0.      13.6129  80.9954 236.1381 337.3401 236.1381]
  Max absolute difference: 0.000000


In [14]:
# Visual comparison: hand-rolled vs PyMC-Marketing
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
weeks = np.arange(20)

# Geometric comparison
ax = axes[0]
ax.plot(weeks, hand_geo, 'o-', color=COLORS[0], linewidth=2.5, markersize=7, label='Hand-rolled')
ax.plot(weeks, pmm_geo, 's--', color=COLORS[3], linewidth=2, markersize=6, alpha=0.8, label='PyMC-Marketing')
ax.set_title('Geometric Adstock: Hand-rolled vs PyMC-Marketing')
ax.set_xlabel('Week')
ax.set_ylabel('Adstocked value')
ax.legend(fontsize=10)

# Delayed comparison
ax = axes[1]
ax.plot(weeks, hand_del, 'o-', color=COLORS[1], linewidth=2.5, markersize=7, label='Hand-rolled')
ax.plot(weeks, pmm_del, 's--', color=COLORS[3], linewidth=2, markersize=6, alpha=0.8, label='PyMC-Marketing')
ax.set_title('Delayed Adstock: Hand-rolled vs PyMC-Marketing')
ax.set_xlabel('Week')
ax.set_ylabel('Adstocked value')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('images/05_pymc_marketing_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
print('Both implementations produce equivalent results.')
print("PyMC-Marketing's transforms are pytensor-based, so they work inside PyMC models for gradient-based inference.")


Both implementations produce equivalent results.
PyMC-Marketing's transforms are pytensor-based, so they work inside PyMC models for gradient-based inference.


---

## 10. Fitting an MMM with Different Adstock Types

Now let's go beyond manually choosing adstock parameters. We'll fit two full **Bayesian MMM** models using PyMC-Marketing — one with geometric adstock and one with delayed adstock — and let the data inform the decay and theta parameters.

In [15]:
from pymc_marketing.mmm import MMM, GeometricAdstock, DelayedAdstock, TanhSaturation
import arviz as az

# Prepare the data
channels = ['tv_spend', 'facebook_spend', 'google_search_spend', 'radio_spend', 'print_spend']
X = df[['date'] + channels].copy()
y = df['revenue'].copy()

print(f"Data: {len(X)} weeks, {len(channels)} channels")
print(f"Revenue range: ${y.min():,.0f} - ${y.max():,.0f}")
print(f"Channels: {channels}")


Data: 104 weeks, 5 channels
Revenue range: $388,468 - $517,165
Channels: ['tv_spend', 'facebook_spend', 'google_search_spend', 'radio_spend', 'print_spend']


In [16]:
# --- Model 1: Geometric Adstock ---
mmm_geo = MMM(
    date_column='date',
    channel_columns=channels,
    adstock=GeometricAdstock(l_max=8),
    saturation=TanhSaturation(),
    yearly_seasonality=2,
)

print("Fitting MMM with GeometricAdstock (l_max=8)...")
print("This may take a few minutes...")
idata_geo = mmm_geo.fit(
    X, y,
    chains=2, draws=500, tune=500, cores=1,
    random_seed=42,
    progressbar=True,
)
print("Geometric model fitted successfully!")


Fitting MMM with GeometricAdstock (l_max=8)...
This may take a few minutes...


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [intercept, adstock_alpha, saturation_b, saturation_c, gamma_fourier, y_sigma]


Output()

...

We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Output()

Geometric model fitted successfully!


In [17]:
# --- Model 2: Delayed Adstock ---
mmm_del = MMM(
    date_column='date',
    channel_columns=channels,
    adstock=DelayedAdstock(l_max=8),
    saturation=TanhSaturation(),
    yearly_seasonality=2,
)

print("Fitting MMM with DelayedAdstock (l_max=8)...")
print("This may take a few minutes...")
idata_del = mmm_del.fit(
    X, y,
    chains=2, draws=500, tune=500, cores=1,
    random_seed=42,
    progressbar=True,
)
print("Delayed adstock model fitted successfully!")


Fitting MMM with DelayedAdstock (l_max=8)...
This may take a few minutes...


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [intercept, adstock_alpha, adstock_theta, saturation_b, saturation_c, gamma_fourier, y_sigma]


Output()

...

We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Output()

Delayed adstock model fitted successfully!


In [18]:
# Compare posterior distributions of adstock decay (alpha) parameters
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for i, ch in enumerate(channels[:3]):
    # Geometric alpha
    ax = axes[0, i]
    geo_alpha = idata_geo.posterior['adstock_alpha'].sel(channel=ch).values.flatten()
    ax.hist(geo_alpha, bins=40, color=COLORS[0], alpha=0.7, density=True, edgecolor='white')
    ax.axvline(np.median(geo_alpha), color=COLORS[0], linestyle='--', linewidth=2, 
               label=f'Median: {np.median(geo_alpha):.3f}')
    ch_label = ch.replace('_spend', '').replace('_', ' ').title()
    ax.set_title(f'{ch_label}\n(Geometric)', fontsize=11)
    ax.set_xlabel('alpha (decay)')
    ax.legend(fontsize=9)
    
    # Delayed alpha
    ax = axes[1, i]
    del_alpha = idata_del.posterior['adstock_alpha'].sel(channel=ch).values.flatten()
    ax.hist(del_alpha, bins=40, color=COLORS[1], alpha=0.7, density=True, edgecolor='white')
    ax.axvline(np.median(del_alpha), color=COLORS[1], linestyle='--', linewidth=2,
               label=f'Median: {np.median(del_alpha):.3f}')
    ax.set_title(f'{ch_label}\n(Delayed)', fontsize=11)
    ax.set_xlabel('alpha (decay)')
    ax.legend(fontsize=9)

plt.suptitle('Posterior Distributions of Adstock Decay (alpha) Parameter', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('images/05_posterior_alpha_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
print('The delayed adstock model estimates different decay rates because it also estimates theta (peak delay).')


The delayed adstock model estimates different decay rates because it also estimates theta (peak delay).


In [19]:
# Show the theta (peak delay) posteriors from the delayed model
fig, axes = plt.subplots(1, len(channels), figsize=(16, 4))

for i, ch in enumerate(channels):
    ax = axes[i]
    theta_samples = idata_del.posterior['adstock_theta'].sel(channel=ch).values.flatten()
    ax.hist(theta_samples, bins=30, color=COLORS[i % len(COLORS)], alpha=0.7, density=True, edgecolor='white')
    median_theta = np.median(theta_samples)
    ax.axvline(median_theta, color='black', linestyle='--', linewidth=2,
               label=f'Median: {median_theta:.1f}')
    ax.set_title(ch.replace('_spend', '').replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('theta (peak delay in weeks)')
    ax.legend(fontsize=8)

plt.suptitle('Posterior Distributions of Peak Delay (theta) \u2014 Delayed Adstock Model', fontweight='bold', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('images/05_posterior_theta.png', dpi=180, bbox_inches='tight')
plt.show()

# Print summary table
print("\nEstimated peak delays (median theta, in weeks):")
print("-" * 45)
for ch in channels:
    theta_vals = idata_del.posterior['adstock_theta'].sel(channel=ch).values.flatten()
    name = ch.replace('_spend', '').replace('_', ' ').title()
    print(f"  {name:20s}: {np.median(theta_vals):.1f} weeks (94% HDI: [{np.percentile(theta_vals, 3):.1f}, {np.percentile(theta_vals, 97):.1f}])")



Estimated peak delays (median theta, in weeks):
---------------------------------------------
  Tv                  : 1.0 weeks (94% HDI: [0.1, 1.8])
  Facebook            : 0.7 weeks (94% HDI: [0.2, 1.2])
  Google Search       : 0.5 weeks (94% HDI: [0.0, 1.8])
  Radio               : 0.8 weeks (94% HDI: [0.1, 2.8])
  Print               : 0.5 weeks (94% HDI: [0.0, 1.6])


---

## 11. Comparing Model Fit

Which adstock specification fits the data better? Let's compare in-sample predictions and compute key accuracy metrics.

In [20]:
# Generate in-sample predictions from both models
# MMM.predict() returns the mean prediction as a numpy array
pred_geo = mmm_geo.predict(X)
pred_del = mmm_del.predict(X)

y_actual = y.values

# Compute metrics
def compute_metrics(y_true, y_pred):
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {'MAPE': mape, 'R2': r2, 'RMSE': rmse}

metrics_geo = compute_metrics(y_actual, pred_geo)
metrics_del = compute_metrics(y_actual, pred_del)

print('Model Comparison: In-Sample Fit')
print('=' * 55)
print(f"{'Metric':<12} {'Geometric':>15} {'Delayed':>15} {'Winner':>10}")
print('-' * 55)
for metric in ['MAPE', 'R2', 'RMSE']:
    g = metrics_geo[metric]
    d = metrics_del[metric]
    if metric == 'MAPE':
        winner = 'Delayed' if d < g else 'Geometric'
        print(f'{metric:<12} {g:>14.2f}% {d:>14.2f}% {winner:>10}')
    elif metric == 'R2':
        winner = 'Delayed' if d > g else 'Geometric'
        print(f'{metric:<12} {g:>15.4f} {d:>15.4f} {winner:>10}')
    else:
        winner = 'Delayed' if d < g else 'Geometric'
        print(f'{metric:<12} {g:>13,.0f} {d:>13,.0f} {winner:>10}')


Sampling: [y]


Output()

Sampling: [y]


Output()

Model Comparison: In-Sample Fit
Metric             Geometric         Delayed     Winner
-------------------------------------------------------
MAPE                   2.72%           2.73%  Geometric
R2                    0.5275          0.5367    Delayed
RMSE                17,154        16,986    Delayed


In [21]:
# Visual comparison of predictions
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Top: time series overlay
ax = axes[0]
ax.plot(df['date'], y_actual / 1000, 'o', color='#94A3B8', markersize=4, alpha=0.6, label='Actual revenue')
ax.plot(df['date'], pred_geo / 1000, '-', color=COLORS[0], linewidth=2,
        label=f'Geometric (R\u00b2={metrics_geo["R2"]:.3f}, MAPE={metrics_geo["MAPE"]:.1f}%)')
ax.plot(df['date'], pred_del / 1000, '-', color=COLORS[1], linewidth=2,
        label=f'Delayed (R\u00b2={metrics_del["R2"]:.3f}, MAPE={metrics_del["MAPE"]:.1f}%)')
ax.set_ylabel('Revenue ($K)')
ax.set_title('In-Sample Predictions: Geometric vs. Delayed Adstock MMM', fontweight='bold')
ax.legend(fontsize=10)

# Bottom: residuals
ax = axes[1]
resid_geo = (y_actual - pred_geo) / 1000
resid_del = (y_actual - pred_del) / 1000
ax.bar(df['date'] - pd.Timedelta(days=1.5), resid_geo, width=2.5, color=COLORS[0], alpha=0.5, label='Geometric residuals')
ax.bar(df['date'] + pd.Timedelta(days=1.5), resid_del, width=2.5, color=COLORS[1], alpha=0.5, label='Delayed residuals')
ax.axhline(0, color='#64748B', linewidth=1, linestyle='-')
ax.set_ylabel('Residual ($K)')
ax.set_xlabel('Date')
ax.set_title('Prediction Residuals')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('images/05_model_fit_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

print(f"\nGeometric adstock MAPE: {metrics_geo['MAPE']:.2f}% | R\u00b2: {metrics_geo['R2']:.4f}")
print(f"Delayed adstock MAPE:   {metrics_del['MAPE']:.2f}% | R\u00b2: {metrics_del['R2']:.4f}")



Geometric adstock MAPE: 2.72% | R²: 0.5275
Delayed adstock MAPE:   2.73% | R²: 0.5367


---

## Summary

| Concept | Key takeaway |
|---|---|
| **Geometric adstock** | Peak at exposure, exponential decay. Best for digital/search. |
| **Delayed adstock** | Gaussian peak at $t=\theta$. Best for TV, print, OOH. |
| **Decay ($\lambda$)** | Controls width of the kernel. Higher = longer-lasting effect. |
| **Theta ($\theta$)** | Controls when the peak occurs. Higher = more delayed. |
| **Half-life** | Common metric to compare adstock types: periods to 50% of peak. |
| **Priors** | Beta for decay, HalfNormal for theta. Tighter for digital, wider for traditional media. |
| **PyMC-Marketing API** | `geometric_adstock()` and `delayed_adstock()` match hand-rolled implementations. |
| **MMM integration** | `GeometricAdstock(l_max=8)` and `DelayedAdstock(l_max=8)` plug directly into `MMM()`. |

## Next Steps

- **[Notebook 08: Tanh Saturation Deep Dive](08-tanh-saturation-deep-dive.ipynb)** — After adstock comes saturation. Learn how `tanh(x / (scalar * alpha))` models diminishing returns.
- **[Notebook 02: Smart Priors from Data](02-smart-priors-from-data.ipynb)** — Auto-generate priors for all model parameters including adstock.

**Core concepts:**
- [Adstock Effects](../docs/core-concepts/adstock-effects.md) — Full documentation on adstock theory and implementation
- [Saturation Curves](../docs/core-concepts/saturation-curves.md) — The next transform in the pipeline after adstock
- [Priors and Distributions](../docs/core-concepts/priors-and-distributions.md) — Choosing priors for decay and theta parameters